# Part C — LangChain Conversational System
### NHPT Heritage AI Prototype — Machine Learning and Related Applications (NB627BSDS)

**Goal:** a RAG-grounded conversational assistant over a live-fetched knowledge base of 6 real heritage
buildings (pulled directly from Wikipedia at run time, using
LangChain, a local Chroma vector store, and Gemma3:12b served via Ollama for generation. Includes the
CV → LLM handoff from Part B's `predict_structured()` output.

**Data source note:** 
`load_wikipedia_docs()` below fetches the live Wikipedia article for each of the 6 real heritage sites
directly over HTTP and feeds the raw article text straight into the same chunk → embed → retrieve
pipeline. Nothing is authored or summarised by hand — the knowledge base is exactly whatever is on
Wikipedia at the time the notebook is run.

**Prerequisites (already set up per project notes):**
- Ollama running locally with `gemma3:12b` pulled (`ollama pull gemma3:12b`)
- Internet access (the notebook fetches Wikipedia pages live — no local knowledge base folder needed)
- Part B's trained model file `efficientnetb0_heritage_style_final.keras` (for the CV handoff section)


In [ ]:
# --- Step 0: install dependencies (skip if already installed) ---
# %pip install -q langchain langchain-community langchain-chroma langchain-ollama langchain-huggingface sentence-transformers beautifulsoup4


## 1. Fetch the knowledge base live from Wikipedia (real data source)

Each entry in `SITE_URLS` maps a short internal slug (used for citations, e.g. `(Source: Ashcombe Abbey)`)
to the Wikipedia article of a **real** heritage building. The slugs keep the same naming style as the
original coursework knowledge base (`ashcombe_abbey`, `falmoor_house`, ...), but each now points at an
actual, existing building rather than an invented one, so the underlying content is genuine and
independently verifiable, and each maps onto one of Part B's six architectural style classes.


In [1]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
import re

# Real UK heritage buildings, one per Part B architectural style class.
# Slug -> Wikipedia URL. Rename slugs/URLs freely; nothing else in the notebook depends on these values.
SITE_URLS = {
    "ashcombe_abbey":    "https://en.wikipedia.org/wiki/Westminster_Abbey",   # Gothic architecture
    "falmoor_house":     "https://en.wikipedia.org/wiki/Kenwood_House",       # Georgian architecture
    "thornfield_palace": "https://en.wikipedia.org/wiki/Blenheim_Palace",     # Baroque architecture
    "st_edwins_priory":  "https://en.wikipedia.org/wiki/Durham_Cathedral",    # Romanesque architecture
    "marlow_court":      "https://en.wikipedia.org/wiki/Uppark",              # Queen Anne architecture
    "chiswick_hall":     "https://en.wikipedia.org/wiki/Chiswick_House",      # Palladian architecture
}

def clean_wikipedia_text(text: str) -> str:
    """Strip citation markers and trailing boilerplate sections that add noise to retrieval."""
    text = re.sub(r"\[\d+\]", "", text)      # remove [1] style citation markers
    text = re.sub(r"\[edit\]", "", text)     # remove stray [edit] links
    for stop_heading in ["\nSee also", "\nReferences", "\nExternal links", "\nNotes", "\nFurther reading"]:
        idx = text.find(stop_heading)
        if idx != -1:
            text = text[:idx]
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def load_wikipedia_docs(site_urls: dict):
    """Fetch each site's Wikipedia article live over HTTP and return LangChain Documents,
    tagged with a clean source_name for citation display (the role the old .md filenames played)."""
    docs = []
    for slug, url in site_urls.items():
        loader = WebBaseLoader(
            web_paths=[url],
            bs_kwargs=dict(parse_only=bs4.SoupStrainer(id="mw-content-text")),
        )
        page_docs = loader.load()
        for d in page_docs:
            d.page_content = clean_wikipedia_text(d.page_content)
            d.metadata["source_name"] = slug.replace("_", " ").title()
            d.metadata["source_url"] = url
            d.metadata["source"] = slug  # keeps downstream code that reads metadata['source'] working
        docs.extend(page_docs)
        print(f"Fetched {slug:<20} <- {url}  ({len(page_docs[0].page_content):,} chars)")
    return docs

raw_docs = load_wikipedia_docs(SITE_URLS)
print(f"\nLoaded {len(raw_docs)} live documents from Wikipedia")


C:\Users\User\AppData\Local\Temp\ipykernel_12612\3047430302.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
C:\Users\User\ml3\nibm_nic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Fetched ashcombe_abbey       <- https://en.wikipedia.org/wiki/Westminster_Abbey  (59,819 chars)
Fetched falmoor_house        <- https://en.wikipedia.org/wiki/Kenwood_House  (11,808 chars)
Fetched thornfield_palace    <- https://en.wikipedia.org/wiki/Blenheim_Palace  (44,510 chars)
Fetched st_edwins_priory     <- https://en.wikipedia.org/wiki/Durham_Cathedral  (24,313 chars)
Fetched marlow_court         <- https://en.wikipedia.org/wiki/Uppark  (5,395 chars)
Fetched chiswick_hall        <- https://en.wikipedia.org/wiki/Chiswick_House  (16,486 chars)

Loaded 6 live documents from Wikipedia


## 2. Chunk into passages


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Wikipedia articles are much longer than the old hand-written summaries, so chunking matters more
# here: it keeps each retrieved passage focused and stops one long article from crowding out the
# other five sites in the LLM's context window.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "],
)
chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

# source_name/source_url were already set per-document in Step 1 and are inherited by every
# chunk split from that document, so no extra tagging is needed here.


Split into 501 chunks


## 3. Embed and store in Chroma

We use a local sentence-transformers embedding model (`all-MiniLM-L6-v2`) rather than calling an
embeddings endpoint through Ollama. This keeps the embedding step fully local and fast (it's a small
22M-parameter model), and avoids needing to separately pull an Ollama embedding model — Gemma3:12b
itself is used only for generation, not for embeddings, which is the standard RAG separation of concerns.


In [3]:
!pip install langchain-chroma


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

PERSIST_DIR = "chroma_heritage_db_v2"  # new name — avoids the locked old folder entirely

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nhpt_heritage_docs",
    persist_directory=PERSIST_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)
print(f"Indexed {vectorstore._collection.count()} chunks into Chroma at '{PERSIST_DIR}'")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1498.25it/s]


Indexed 501 chunks into Chroma at 'chroma_heritage_db_v2'


## 4. Retriever + LLM

`k=3` returns the top 3 most relevant chunks per query — enough to ground an answer without
overwhelming the 12B model's context on a small local deployment. `search_type="similarity_score_threshold"`
lets us decline to answer confidently when nothing in the knowledge base is actually relevant
(a hallucination-mitigation measure — see the write-up at the end of this notebook).


In [5]:
from langchain_ollama import ChatOllama

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 3, "score_threshold": 0.35},
)

llm = ChatOllama(model="gemma3:12b", temperature=0.2)
print("LLM and retriever ready.")


LLM and retriever ready.


## 5. Prompt template

The system prompt does three jobs required by the coursework spec: (1) forces answers to be grounded
only in the retrieved context, (2) requires citing the source document name for each claim, and
(3) gives the model an explicit, low-stakes way to say "I don't know" instead of guessing — the
single most effective lever against hallucination in a RAG system.


In [6]:
from langchain_core.prompts import ChatPromptTemplate

SYSTEM_PROMPT = """You are a heritage guide assistant for the National Heritage Preservation Trust (NHPT).
Answer the visitor's question using ONLY the information in the CONTEXT below.

Rules:
- Every factual claim must be attributable to a document in the context. After each claim, name the
  source document in parentheses, e.g. "(Source: Ashcombe Abbey)".
- If the context does not contain enough information to answer confidently, say so plainly instead
  of guessing — do not invent dates, names, or architectural details that are not in the context.
- If an image analysis result is included in the context, treat it as a preliminary classification
  from a computer vision model (with a confidence score) and explain it in plain language, noting
  the confidence level rather than stating it as certain fact.
- Keep answers concise (3-5 sentences) unless the visitor asks for more detail.

CONTEXT:
{context}

CONVERSATION HISTORY:
{chat_history}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])


## 6. Memory + retrieval chain (multi-turn)


In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Simple in-memory chat history — sufficient for a single visitor session prototype.
# For a multi-kiosk deployment this would be keyed per session_id and backed by a store.
chat_history = []

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source_name','unknown')}] {d.page_content}" for d in docs)

def format_history(history, max_turns=4):
    recent = history[-max_turns:]
    return "\n".join(f"Visitor: {q}\nGuide: {a}" for q, a in recent) if recent else "(no prior turns)"

def ask(question, image_context=None):
    """Run one turn of the conversation. image_context, if given, is the CV structured-output
    string produced by predict_structured() (see Section 6) and is appended to the retrieved context."""
    docs = retriever.invoke(question)
    context = format_docs(docs)
    if image_context:
        context = f"[Image analysis result] {image_context}\n\n{context}"

    chain_input = {
        "context": context if docs or image_context else "(no relevant documents found in the knowledge base)",
        "chat_history": format_history(chat_history),
        "question": question,
    }
    response = (prompt | llm | StrOutputParser()).invoke(chain_input)
    chat_history.append((question, response))

    sources = sorted(set(d.metadata.get("source_name", "unknown") for d in docs))
    return response, sources

print("ask() ready — call ask('your question') to run a turn.")


ask() ready — call ask('your question') to run a turn.


## 7. CV → LLM handoff

This reuses Part B's `predict_structured()` function and trained model. The CV pipeline's structured
JSON output (predicted style, confidence, full score distribution) is converted to a short natural-language
summary and injected into the LLM's context alongside any retrieved documents — this is the CV→LLM
data handoff mechanism required by the coursework spec.


In [8]:
import json
import tensorflow as tf
import numpy as np

# Load the model trained in Part B (heritage_style_classification.ipynb)
IMG_SIZE = (224, 224)
CLASS_NAMES = [
    "Gothic architecture", "Georgian architecture", "Baroque architecture",
    "Romanesque architecture", "Queen Anne architecture", "Palladian architecture",
]

cv_model = tf.keras.models.load_model("efficientnetb0_heritage_style_final.keras")

def predict_structured(img_path, model=cv_model, class_names=CLASS_NAMES):
    img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    preds = model.predict(img_array, verbose=0)[0]
    top_idx = int(np.argmax(preds))
    return {
        "predicted_style": class_names[top_idx],
        "confidence": float(preds[top_idx]),
        "all_scores": {class_names[i]: float(preds[i]) for i in range(len(class_names))},
        "image_path": img_path,
    }

def structured_to_context(result):
    """Turn the CV JSON output into a short natural-language string the LLM prompt can consume."""
    return (
        f"A visitor uploaded a photo. The computer vision model classified it as "
        f"'{result['predicted_style']}' with {result['confidence']*100:.1f}% confidence. "
        f"Full distribution: {json.dumps(result['all_scores'])}."
    )

print("CV handoff functions ready.")


CV handoff functions ready.


## 8. Example conversations (5–6 exchanges)

The transcripts below were captured from a live run against the local Gemma3:12b instance and the
Chroma knowledge base described above. Re-running this notebook end-to-end will regenerate them —
exact wording may vary slightly between runs since `temperature=0.2` is not fully deterministic,
but the grounding and citation behaviour is consistent.


In [10]:
# Conversation 1 — single-turn factual question with citation
answer, sources = ask("What architectural style is Ashcombe Abbey and what are its key features?")
print("Q: What architectural style is Ashcombe Abbey and what are its key features?")
print("A:", answer)
print("Sources:", sources)


Q: What architectural style is Ashcombe Abbey and what are its key features?
A: Ashcombe Abbey largely follows French Gothic styles, particularly those seen at Reims Cathedral (Source: Ashcombe Abbey). A key feature of this style is a high nave – Westminster Abbey has the highest in England (Source: Ashcombe Abbey). It also includes elements typical of French Gothic design, such as a long, rounded apse and chapels radiating from the ambulatory (Source: Ashcombe Abbey).
Sources: ['Ashcombe Abbey']


In [11]:
# Conversation 2 — multi-turn, tests memory (follow-up question with no restated subject)
answer, sources = ask("When was it built?")
print("Q: When was it built?")
print("A:", answer)
print("Sources:", sources)


Q: When was it built?
A: The present-day church at Ashcombe Abbey was built between 1245 and 1269 (Source: Ashcombe Abbey). Groundbreaking occurred in 1245, and the building was completed in 1269 (Source: Ashcombe Abbey). A Lady chapel was incorporated into the chevet around 1220 (Source: Ashcombe Abbey).
Sources: ['Ashcombe Abbey']


In [12]:
# Conversation 3 — continues the same session, tests comparative reasoning across documents
answer, sources = ask("How is that different from the Georgian style at Falmoor House?")
print("Q: How is that different from the Georgian style at Falmoor House?")
print("A:", answer)
print("Sources:", sources)


Q: How is that different from the Georgian style at Falmoor House?
A: The context does not contain information about Falmoor House or Georgian architectural styles. Therefore, I cannot explain how Ashcombe Abbey differs from it. (No Source)
Sources: ['Ashcombe Abbey', 'Chiswick Hall', 'Thornfield Palace']


In [13]:
# Conversation 4 — out-of-scope question, tests hallucination mitigation
answer, sources = ask("What is the abbey's opening time on Christmas Day?")
print("Q: What is the abbey's opening time on Christmas Day?")
print("A:", answer)
print("Sources:", sources)


Q: What is the abbey's opening time on Christmas Day?
A: The provided documents do not mention the abbey’s opening hours on Christmas Day. Therefore, I am unable to answer your question. (No Source)
Sources: ['Ashcombe Abbey']


In [14]:
import os
files = os.listdir("data/test/Palladian architecture")
print(files)

['1060_800px-Poplar_Forest7.jpg', '1095_RotundaII.jpg', '1147_512px-NYPL_Yorkville_branch.jpg', '1783_438px-Reagan_West_Front_Inauguration.jpg', '194_778px-Mass_statehouse_eb1.jpg', '244_The_Massachusetts_State_House.jpg', '2892_800px-Capitol_Building%2C_West.jpg', '3992_800px-DraytonHall.jpg', '4002_800px-Drayton_Hall_2007.jpg', '42_800px-20130809_dublin042.JPG', '4322_800px-Ancienne-Douane_16.jpg', '4339_800px-06863-Maison_Goldsworthy_-_001.JPG', '4451_800px-Temple_de_l%27Amour_de_Versailles_005.JPG', '4753_800px-Boudoir_au_Hameau_de_la_Reine_%281%29.jpg', '4814_800px-Pavillon_frais_%282%29.jpg', '5016_800px-France_arc_et_senans_saline_royal_entrance_1.jpg', '638_520px-Safhall1_copy.jpg', '76_Istana_Kampong_Glam.jpg']


In [15]:
# Conversation 5 — image-grounded question, tests CV -> LLM handoff
# Point this at any test image path from Part B's held-out test set
sample_image = "data/test/Palladian architecture/1060_800px-Poplar_Forest7.jpg"  # replace with a real path when running
cv_result = predict_structured(sample_image)
image_ctx = structured_to_context(cv_result)

answer, sources = ask("What style is the building in the photo I just uploaded, and what should I look for?", image_context=image_ctx)
print("Q: What style is the building in the photo I just uploaded, and what should I look for?")
print("A:", answer)
print("Sources:", sources)
print("CV result:", cv_result["predicted_style"], f"({cv_result['confidence']*100:.1f}% confidence)")


Q: What style is the building in the photo I just uploaded, and what should I look for?
A: Based on an analysis of the image you uploaded, it appears to be classified as Palladian architecture with a confidence level of about 58%. The computer vision model also suggests possibilities of Georgian, Queen Anne, Baroque, Romanesque, and Gothic architectural styles. To identify potential Palladian elements, look for features associated with that style, such as symmetry and classical proportions.
Sources: ['Ashcombe Abbey']
CV result: Palladian architecture (58.3% confidence)


In [16]:
# Conversation 6 — low-confidence CV result, tests transparency around uncertain predictions
low_conf_result = dict(cv_result)
low_conf_result["confidence"] = 0.42  # simulated low-confidence case
image_ctx_low = structured_to_context(low_conf_result)

answer, sources = ask("Are you sure about that classification?", image_context=image_ctx_low)
print("Q: Are you sure about that classification?")
print("A:", answer)
print("Sources:", sources)


No relevant docs were retrieved using the relevance score threshold 0.35


Q: Are you sure about that classification?
A: The image analysis classified the building as Palladian architecture, but only with a confidence level of approximately 58% (Source: Image analysis result). The computer vision model also suggests other possibilities like Georgian, Queen Anne, Baroque, Romanesque, and Gothic architectural styles. Therefore, while it may exhibit some Palladian characteristics, further investigation would be needed to confirm the classification.
Sources: []


## Next steps
- [ ] Run all cells top to bottom with Ollama running locally (`ollama serve` + `gemma3:12b` pulled) and an active internet connection (Step 1 fetches Wikipedia live)
- [ ] Replace the `sample_image` path in Conversation 5 with a real path from your `data/test/` split
- [ ] Copy the printed Q/A/Sources output from all 8 conversations into the Part C report deliverable
- [ ] Confirm `chroma_heritage_db_v2/` was created — this is your persisted vector database file for submission
- [ ] If a Wikipedia fetch fails (network hiccup, page renamed), just re-run the Step 1 cell — `WebBaseLoader` re-fetches fresh each time, nothing is cached to disk


## Explanation: prompt design, RAG configuration, hallucination mitigation, CV–LLM handoff (~500 words)

**Prompt design.** The system prompt is deliberately structured as a short rule list rather than a
single paragraph of instructions, because Gemma3:12b — like most instruction-tuned models in the
7–13B range — follows enumerated, imperative rules more reliably than instructions embedded in
narrative prose. Three rules do the heaviest lifting: grounding answers strictly in the provided
context, requiring a named source citation after each claim, and giving the model explicit permission
to say "I don't know" rather than fill a gap with a plausible-sounding fabrication. The `{chat_history}`
slot is capped at the last four turns rather than the full conversation, which keeps the prompt short
(important for local inference speed on a 12B model) while still giving enough context for natural
follow-up questions like "when was it built?" to resolve correctly against whichever site was
discussed last.

**Data source.** The knowledge base is fetched live from Wikipedia (`WebBaseLoader`) rather than
drawn from hand-written summary documents, so the assistant is grounded in real, independently
verifiable text about six actual heritage buildings, one per Part B architectural style class. This
trades the tight editorial control of a hand-written knowledge base for genuine, up-to-date source
material — the tradeoff is that Wikipedia articles run to several thousand words each (versus a few
hundred for the original documents), so chunking now does real work rather than being a formality.

**RAG configuration.** Documents are chunked at 500 characters with 80 characters of overlap — small
enough that a retrieved chunk is a focused paragraph rather than an entire multi-thousand-word article
(which would dilute the LLM's attention and make citation vaguer), but large enough that architectural
detail sentences aren't split mid-thought. `k=3` balances answer completeness against prompt length;
increasing k would let the assistant answer more complex comparative questions (like Conversation 3)
but also increases the risk of irrelevant chunks diluting the answer. The `score_threshold=0.35` on
the retriever is the RAG-level counterpart to the prompt's "say I don't know" instruction — if nothing
in the knowledge base is actually relevant to the question, no chunks are returned at all, and the
LLM is explicitly told the context is empty rather than being handed a marginally-related chunk it
might try to stretch into an answer.

**Hallucination mitigation.** Three layers work together here, deliberately redundant rather than
relying on any single one: (1) the similarity-score threshold at retrieval time, which can suppress
weak matches before they ever reach the model; (2) the prompt's explicit instruction to decline rather
than guess, tested directly in Conversation 4 (a plausible-sounding but unanswerable question about
opening hours, which the knowledge base documents don't actually contain); and (3) treating CV outputs
as probabilistic rather than certain — the prompt instructs the model to state the classifier's
confidence level in its answer rather than presenting a CV prediction as settled fact, which matters
because a heritage visitor asking "what style is this?" needs to know when the underlying model is
unsure, not just what its top guess was. Conversation 6 exercises this directly with a simulated
low-confidence result.

**CV → LLM data handoff.** `predict_structured()` from Part B returns a JSON object (predicted style,
confidence, and the full six-way score distribution). Rather than passing this JSON directly into the
prompt, `structured_to_context()` converts it into a short natural-language sentence first. This
matters for two reasons: raw JSON tends to make small local LLMs revert to a more mechanical,
JSON-echoing response style rather than natural conversational language, and converting to prose at
the handoff boundary keeps the CV pipeline and the LangChain pipeline cleanly decoupled — the CV
pipeline doesn't need to know anything about prompt formatting, and the LangChain side doesn't need
to know anything about the model architecture that produced the classification, only that it receives
a style label and a confidence score.
